# 🤖 Module 6 — Introduction to Transformers
### Introduction to Deep Learning | AgenticLabs.ng

---

## 🎯 Learning Objectives
- Understand the key limitation of RNNs that Transformers solve
- Deeply understand **self-attention**: the core mechanism
- Understand **positional encoding** and why it is needed
- Map the full **Transformer architecture** (encoder + decoder)
- Use a **pretrained BERT model** from Hugging Face for text classification
- Understand what **GPT** is and how it differs from BERT
- Connect transformers to the broader AI ecosystem (LLMs, Vision Transformers)

---

## 6.1 — The Problem with RNNs (and Why Transformers Won)

RNNs process sequences **step by step** — each token must wait for all previous tokens. This means:

1. **Cannot parallelise** during training → slow
2. **Long-range dependencies are hard** — information must travel through many timesteps
3. **Bottleneck**: all context compressed into a fixed-size hidden vector

The **Transformer** (Vaswani et al., 2017 — "Attention Is All You Need") solved this by replacing recurrence with **self-attention**, allowing every token to attend directly to every other token in a single operation.


## 6.2 — Self-Attention: The Core Idea

For each token, self-attention computes how much it should **attend to** every other token in the sequence.

```
Attention(Q, K, V) = softmax(QK^T / √d_k) × V
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ── Self-attention from scratch ────────────────────────────
class SelfAttention(nn.Module):
    """Single-head self-attention"""
    def __init__(self, embed_dim):
        super().__init__()
        self.d_k = embed_dim
        self.W_Q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_K = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_V = nn.Linear(embed_dim, embed_dim, bias=False)
    
    def forward(self, x, return_weights=False):
        """x: (batch, seq_len, embed_dim)"""
        Q = self.W_Q(x)         # Queries
        K = self.W_K(x)         # Keys
        V = self.W_V(x)         # Values
        
        # Scaled dot-product attention
        scores  = (Q @ K.transpose(-2, -1)) / (self.d_k ** 0.5)
        weights = F.softmax(scores, dim=-1)
        output  = weights @ V
        
        if return_weights:
            return output, weights
        return output

# ── Demo with a toy sentence ──────────────────────────────
tokens = ["The", "cat", "sat", "on", "the", "mat"]
seq_len = len(tokens)
embed_dim = 16

# Random embeddings (in practice, learned)
x = torch.randn(1, seq_len, embed_dim)

attn = SelfAttention(embed_dim)
out, weights = attn(x, return_weights=True)

print(f"Input shape       : {x.shape}        → (batch, seq, embed)")
print(f"Output shape      : {out.shape}     → same shape")
print(f"Attention weights : {weights.shape} → (batch, seq, seq)")

# Visualise attention weights
weights_np = weights.squeeze(0).detach().numpy()
plt.figure(figsize=(7, 5))
sns.heatmap(weights_np, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=tokens, yticklabels=tokens, linewidths=0.5)
plt.title("Self-Attention Weights
(each row = how much each token attends to others)")
plt.xlabel("Keys (attended to)"); plt.ylabel("Queries (attending from)")
plt.tight_layout(); plt.show()


## 6.3 — Multi-Head Attention

Instead of one attention operation, we run **h** attention heads in parallel, each learning to attend to different aspects of the input (e.g. one head for syntax, one for semantics, one for coreference).

In [ ]:
# ── Multi-head attention ─────────────────────────────────
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.d_k       = embed_dim // num_heads
        
        self.W_Q = nn.Linear(embed_dim, embed_dim)
        self.W_K = nn.Linear(embed_dim, embed_dim)
        self.W_V = nn.Linear(embed_dim, embed_dim)
        self.W_O = nn.Linear(embed_dim, embed_dim)
    
    def split_heads(self, x):
        B, T, D = x.shape
        return x.view(B, T, self.num_heads, self.d_k).transpose(1, 2)
    
    def forward(self, x):
        B, T, D = x.shape
        Q = self.split_heads(self.W_Q(x))
        K = self.split_heads(self.W_K(x))
        V = self.split_heads(self.W_V(x))
        
        scores  = (Q @ K.transpose(-2,-1)) / (self.d_k**0.5)
        weights = F.softmax(scores, dim=-1)
        out     = (weights @ V).transpose(1,2).contiguous().view(B, T, D)
        return self.W_O(out), weights

mha = MultiHeadAttention(embed_dim=64, num_heads=8)
x   = torch.randn(1, 6, 64)
out, heads = mha(x)
print(f"Input shape  : {x.shape}")
print(f"Output shape : {out.shape}")
print(f"Heads shape  : {heads.shape}  → (batch, num_heads, seq, seq)")

# Visualise each head's attention pattern
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.ravel()):
    sns.heatmap(heads[0,i].detach().numpy(), ax=ax, cmap='Blues',
                xticklabels=tokens, yticklabels=tokens if i%4==0 else [],
                cbar=False, linewidths=0.3)
    ax.set_title(f"Head {i+1}", fontsize=10)
plt.suptitle("8 Attention Heads — Each Attends Differently", fontsize=13)
plt.tight_layout(); plt.show()


## 6.4 — Positional Encoding

Self-attention is **permutation-invariant** — it doesn't know the order of tokens. We fix this by adding a positional encoding to every token embedding.

In [ ]:
# ── Sinusoidal positional encoding ────────────────────────
def get_positional_encoding(max_len, embed_dim):
    pe = torch.zeros(max_len, embed_dim)
    pos = torch.arange(0, max_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, embed_dim, 2).float() *
                    (-np.log(10000.0) / embed_dim))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

pe = get_positional_encoding(50, 128)
plt.figure(figsize=(12, 4))
plt.subplot(1,2,1)
plt.imshow(pe[:20, :64].T, aspect='auto', cmap='RdBu')
plt.xlabel("Position"); plt.ylabel("Embedding dim")
plt.title("Positional Encoding (first 64 dims)")
plt.colorbar()

plt.subplot(1,2,2)
for pos in [0, 5, 10, 20]:
    plt.plot(pe[pos, :32].numpy(), label=f"Position {pos}")
plt.xlabel("Embedding dimension"); plt.ylabel("Value")
plt.title("Positional Encoding Values Per Position")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 6.5 — The Full Transformer Architecture

```
Input tokens
    ↓
Token Embedding + Positional Encoding
    ↓  (repeated N times)
┌──────────────────────────────────┐
│  Multi-Head Self-Attention       │
│  + Add & Norm (residual)         │
│  Feed-Forward Network (MLP)      │
│  + Add & Norm (residual)         │
└──────────────────────────────────┘
    ↓
Task-specific head (classification / generation)
```

**BERT** = Encoder only → good for understanding tasks (classification, NER, QA)  
**GPT**  = Decoder only → good for generation tasks (text completion, chat)


## 6.6 — Using Pretrained BERT with Hugging Face

This is the most practical skill in Module 6. You will fine-tune BERT for text classification in just a few lines.

In [ ]:
# ── Install Hugging Face Transformers ─────────────────────
!pip install transformers datasets --quiet
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           Trainer, TrainingArguments)
from datasets import load_dataset
import torch

print("✅ Transformers library loaded")
print(f"Transformers version: ", end="")
import transformers; print(transformers.__version__)


In [ ]:
# ── Load tokeniser and model ──────────────────────────────
MODEL_NAME = "distilbert-base-uncased"   # lighter version of BERT

tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME)
model_bert = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Move to device
model_bert = model_bert.to(device)
total_params = sum(p.numel() for p in model_bert.parameters())
print(f"Model     : {MODEL_NAME}")
print(f"Parameters: {total_params/1e6:.1f}M")
print(f"Device    : {device}")

# ── Test tokenisation ─────────────────────────────────────
sample = "This movie was absolutely fantastic! Best film I've seen all year."
tokens = tokeniser(sample, return_tensors='pt', padding=True, truncation=True, max_length=128)
print(f"\nSample    : {sample}")
print(f"Token IDs : {tokens['input_ids'][0][:10]}... (length: {tokens['input_ids'].shape[1]})")
print(f"Decoded   : {tokeniser.convert_ids_to_tokens(tokens['input_ids'][0][:10])}")


In [ ]:
# ── Fine-tune on IMDb ──────────────────────────────────────
# Load a small subset for demo (full fine-tune takes longer in Colab)
raw_dataset = load_dataset("imdb")

def tokenise_fn(examples):
    return tokeniser(examples["text"], truncation=True, max_length=256, padding="max_length")

print("Tokenising dataset...")
tokenised = raw_dataset.map(tokenise_fn, batched=True)
tokenised = tokenised.rename_column("label", "labels")
tokenised.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Use a small subset for quick demo
small_train = tokenised["train"].select(range(2000))
small_test  = tokenised["test"].select(range(500))

print(f"Fine-tuning on {len(small_train)} samples...")

training_args = TrainingArguments(
    output_dir        = "./bert-imdb",
    num_train_epochs  = 2,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    evaluation_strategy = "epoch",
    save_strategy       = "epoch",
    load_best_model_at_end = True,
    logging_dir         = "./logs",
    report_to           = "none",
    fp16                = torch.cuda.is_available(),
)

trainer = Trainer(
    model          = model_bert,
    args           = training_args,
    train_dataset  = small_train,
    eval_dataset   = small_test,
)

trainer.train()
print("\n✅ Fine-tuning complete!")


In [ ]:
# ── Quick inference demo ──────────────────────────────────
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model=model_bert, tokenizer=tokeniser, device=0 if torch.cuda.is_available() else -1)

test_reviews = [
    "An absolute masterpiece. The direction, acting, and script were all flawless.",
    "I fell asleep halfway through. Boring, predictable, and poorly acted.",
    "It had its moments but overall felt rushed and unsatisfying.",
]

print("BERT Sentiment Classification:")
print("=" * 60)
for review in test_reviews:
    result = classifier(review[:512])[0]
    label  = "POSITIVE 😊" if result['label'] == 'LABEL_1' else "NEGATIVE 😞"
    print(f"\n{review[:70]}...")
    print(f"→ {label} | Confidence: {result['score']:.3f}")


## 6.7 — BERT vs GPT: Key Differences

| Property | BERT | GPT |
|---|---|---|
| Architecture | Encoder only | Decoder only |
| Training objective | Masked Language Model | Next token prediction |
| Attention | Bidirectional (sees all tokens) | Causal (left-to-right only) |
| Best for | Understanding (classification, QA) | Generation (chat, completion) |
| Examples | BERT, RoBERTa, DistilBERT | GPT-2, GPT-4, Claude |

## 6.8 — Where to Go Next

You have now completed the full introduction. The frontier from here:

- **Vision Transformers (ViT)**: Apply transformers directly to image patches
- **Diffusion Models**: The technology behind DALL-E, Stable Diffusion
- **LLM Fine-tuning**: LoRA, QLoRA — efficiently tune large models
- **Agents & RAG**: The focus of AgenticLabs' Agentic AI Program

---

## 📝 Exercises

1. **Attention visualisation**: Use `BertViz` library to visualise BERT's attention heads on a sample sentence.
2. **Full fine-tune**: Remove `.select(range(2000))` and fine-tune on the full 25K IMDb training set. What accuracy do you get?
3. **Different task**: Use the same pipeline to fine-tune for a different dataset, like `sst2` (also a sentiment task but shorter sentences).
4. **GPT-2 generation**: Use `from transformers import GPT2LMHeadModel, GPT2Tokenizer` to generate text completions from GPT-2.

---

## ✅ Module 6 Summary

| Concept | Key takeaway |
|---|---|
| Self-attention | Every token attends to every other token in one operation |
| Multi-head attention | Multiple attention patterns learned in parallel |
| Positional encoding | Injects order information since attention is permutation-invariant |
| BERT | Bidirectional encoder, best for understanding tasks |
| GPT | Causal decoder, best for generation tasks |
| Transfer learning | Always fine-tune a pretrained model — never train from scratch for NLP |

**Next → Capstone Project!**
